# Disaster Tweets LLM API Experiment

This notebook is a separate, lightweight experiment for prompt-based classification using an LLM API.

It is intentionally isolated from the RoBERTa notebook so both workflows can run independently.


## 1. Goal

Use a hosted LLM as a zero-shot or few-shot classifier for disaster tweet detection.

Planned output:
- validation F1 on a local train/valid split
- optional Kaggle submission file for `test.csv`


## 2. Key setup

Store your API key in an environment variable or local `.env` file.

Recommended variable name:
- `GROQ_API_KEY`

Do not hardcode the key inside the notebook or commit it to GitHub.


In [ ]:
# Minimal dependencies for the API notebook.
# Install only if needed. This keeps everything inside the notebook, no terminal required.
import importlib.util
import subprocess
import sys

required_packages = ['groq', 'python-dotenv', 'pandas', 'scikit-learn', 'tqdm']
missing_packages = []
for package in required_packages:
    module_name = package.replace('-', '_')
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(package)

if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])

from pathlib import Path
import os
import re

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

try:
    from groq import Groq
except ImportError:
    Groq = None

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'

if load_dotenv is not None:
    load_dotenv()

API_KEY = os.getenv('GROQ_API_KEY')
GROQ_MODEL = os.getenv('GROQ_MODEL', 'llama-3.1-8b-instant')
print('API key loaded:', bool(API_KEY))
print('Groq model:', GROQ_MODEL)
print('groq package available:', Groq is not None)


## 3. Data loading

We reuse the same `train.csv` and `test.csv` files, but this notebook keeps the LLM workflow separate from the RoBERTa notebook.


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH) if TEST_PATH.exists() else None

print(train_df.shape)
if test_df is not None:
    print(test_df.shape)


## 4. Lightweight text cleanup

For LLM prompting we keep cleaning minimal so the model sees natural language.

Suggested cleanup:
- lowercasing
- remove URLs
- normalize whitespace
- limit very long tweets to keep token usage low


In [ ]:
MAX_TEXT_CHARS = 160

def light_clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:MAX_TEXT_CHARS]

train_df['llm_text'] = train_df['text'].apply(light_clean)
if test_df is not None:
    test_df['llm_text'] = test_df['text'].apply(light_clean)

train_df[['text', 'llm_text', 'target']].head()


## 5. Prompt design

Keep the prompt strict and short so token usage stays low.

Recommended response format:
- return only `0` or `1`
- no explanation in the final answer

This helps reduce token usage and makes parsing reliable.


In [ ]:
SYSTEM_PROMPT = (
    'You are a strict binary classifier for disaster tweets. Return only comma-separated 0/1 labels, one label per tweet, with no explanation.'
)

def build_prompt(tweet_texts):
    lines = [
        'Classify each tweet as disaster (1) or non-disaster (0).',
        'Return only comma-separated digits, one digit per tweet, in the same order.',
        'Do not add any words, brackets, code fences, or explanations.',
        'Example output: 0,1,0,0',
    ]
    for i, tweet_text in enumerate(tweet_texts, 1):
        lines.append(f'{i}: {tweet_text}')
    return '\n'.join(lines)


## 6. Batching strategy

To stay within Groq limits, process tweets in small batches and keep the prompt compact.

Recommended conservative starting point:
- batch size: `10`
- run on a validation subset first
- only scale up if token usage is safe


In [ ]:
BATCH_SIZE = 4
VALIDATION_SIZE = 200
MAX_OUTPUT_TOKENS = 32
TEMPERATURE = 0.0

train_part, valid_part = train_test_split(
    train_df,
    test_size=VALIDATION_SIZE,
    random_state=42,
    stratify=train_df['target'],
)

print(train_part.shape, valid_part.shape)


## 7. API inference stub

This cell is intentionally a stub until you add the Groq key.

Once ready, implement one function that sends a small batch of tweets and parses 0/1 predictions.


In [ ]:
if API_KEY is None:
    raise ValueError('GROQ_API_KEY is missing. Put it in .env and rerun the setup cell.')

if Groq is None:
    raise ImportError('groq package is missing. Run the install cell above and rerun the imports.')

client = Groq(api_key=API_KEY)

def parse_label_list(raw_text, expected_len):
    labels = [int(x) for x in re.findall(r'[01]', raw_text)]
    if len(labels) < expected_len:
        raise ValueError(f'Could not parse enough labels from response: {raw_text!r}')
    return labels[:expected_len]

def _call_llm(tweet_texts):
    prompt = build_prompt(tweet_texts)
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=TEMPERATURE,
        max_tokens=max(MAX_OUTPUT_TOKENS, len(tweet_texts) * 4),
    )
    raw_text = response.choices[0].message.content.strip()
    return parse_label_list(raw_text, len(tweet_texts))

def predict_batch_with_llm(tweet_texts):
    if not tweet_texts:
        return []
    try:
        return _call_llm(tweet_texts)
    except Exception as e:
        message = str(e)
        if 'rate_limit_exceeded' in message or 'Request too large' in message or '413' in message or 'Could not parse enough labels' in message:
            if len(tweet_texts) == 1:
                raise
            mid = len(tweet_texts) // 2
            left = predict_batch_with_llm(tweet_texts[:mid])
            right = predict_batch_with_llm(tweet_texts[mid:])
            return left + right
        raise


## 8. Evaluation plan

After predictions are available:

1. compare predictions with `valid_part['target']`
2. compute `F1`
3. inspect classification report and confusion matrix
4. only then scale from validation to `test.csv`


## 9. Run validation and score

This final cell runs the prompt-based classifier on the local validation split and prints the `F1` score.


In [ ]:
valid_predictions = predict_batch_with_llm(valid_part['llm_text'].tolist())
valid_predictions = pd.Series(valid_predictions, index=valid_part.index)

valid_f1 = f1_score(valid_part['target'], valid_predictions)
print(f'Validation F1: {valid_f1:.4f}')
print(classification_report(valid_part['target'], valid_predictions))
print('Confusion matrix:')
print(confusion_matrix(valid_part['target'], valid_predictions))


In [ ]:
all_predictions = []
texts = valid_part['llm_text'].tolist()
for start in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[start:start + BATCH_SIZE]
    batch_predictions = predict_batch_with_llm(batch_texts)
    all_predictions.extend(batch_predictions)

valid_predictions = pd.Series(all_predictions, index=valid_part.index)
valid_f1 = f1_score(valid_part['target'], valid_predictions)
print(f'Validation F1: {valid_f1:.4f}')
print(classification_report(valid_part['target'], valid_predictions))
print('Confusion matrix:')
print(confusion_matrix(valid_part['target'], valid_predictions))
